<a href="https://colab.research.google.com/github/D2himself/hop-specialist/blob/main/01_Distill_Reasoning_Gemma3_270M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Research Objective**: Fine-tuning Gemma-3-270M to act as a specialized decomposition agent for multi-hop queries, distilling logical hop-transition capabilities from GPT-4o.

Model choice (temporary):
This notebook uses `gemma-3-270m-it` to match the Unsloth tutorial. A controlled replication with `gemma-3-270m` (base) will be run to isolate instruction-tuning effects.

- Model (Gemma3-270M)
- Data

In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 13.5 MB/s eta 0:00:00


In [ ]:
import transformers
import trl
import datasets

# Model


In [ ]:
MODEL_NAME = "google/gemma-3-270m-it"

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[INFO] Using device: {device}")

[INFO] Using device: cuda


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype='auto',
    device_map="auto",
    attn_implementation="eager"
)

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


print(f"[INFO] Model on devices: {model.device}")
print(f"[INFO] Model using dtypes: {model.dtype}")

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

[INFO] Model on devices: cuda:0
[INFO] Model using dtypes: torch.bfloat16


In [ ]:


# Model requires numbers (tokens) as input
# turn strings to tokens via a tokenizer

# model("Hello my name is dimeji")

In [ ]:
tokenizer("Hello my name is dimeji.", return_tensors='pt')

{'input_ids': tensor([[     2,   9259,   1041,   1463,    563,  89904,   5573, 236761]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
outputs = model((tokenizer("Hello my name is dimeji.", return_tensors='pt')["input_ids"]).to('cuda'))

outputs.keys()

odict_keys(['logits', 'past_key_values'])

In [ ]:
model

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

### Try the model with a pipeline

In [ ]:
from transformers import pipeline


pipe = pipeline("text-generation",
                model = MODEL_NAME,
                token = tokenizer)

input_text = "Hi my name is Dimeji"

input_prompt = pipe.tokenizer.apply_chat_template(input_text,
                                                  tokenize=False,
                                                  add_generation_prompt=True)

input_prompt

Device set to use cuda:0


TemplateError: Conversation roles must alternate user/assistant/user/assistant/...